# PromQL Zero to Hero · LoRA router

Select **Runtime → Change runtime type → T4 GPU**. This notebook performs a real training run and can consume GPU resources. Outputs are intentionally empty until executed. No provider credentials or production data are required.

The five-class router is a narrow classifier. It cannot award badges or write progress. The small validation set is not an independent generalization benchmark.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

In [ ]:
!git clone https://github.com/sivalinb/promql-zero-to-hero.git
%cd promql-zero-to-hero
%pip install -e ".[training]" "llamafactory>=0.9.3,<1"

## Prepare and inspect synthetic data
Seed families are split before variations. Inspect both classes and split provenance before training.

In [ ]:
!python training/prepare.py
import json
from pathlib import Path

train = json.loads(Path("training/data/train.json").read_text())
validation = json.loads(Path("training/data/validation.json").read_text())
print(len(train), "training examples;", len(validation), "validation examples")
print(json.dumps(train[0], indent=2))

## Measure the base model
This downloads Qwen3-1.7B-Base from its official Hugging Face repository. It may emit invalid JSON before task-specific fine-tuning; those predictions count as errors.

In [ ]:
!python training/evaluate.py --model Qwen/Qwen3-1.7B-Base --output training/data/baseline.json

## Train and merge
Rank-8 LoRA, FP16, one example per device with gradient accumulation, three epochs. The non-thinking template targets short JSON responses. If you change hyperparameters, save the changed configuration and keep an independent test set.

In [ ]:
!llamafactory-cli train training/lora.yaml
!llamafactory-cli export training/merge.yaml

## Measure the merged model on the same validation rows

In [ ]:
!python training/evaluate.py --model training/output/router-merged --output training/data/merged.json
base = json.loads(Path("training/data/baseline.json").read_text())
merged = json.loads(Path("training/data/merged.json").read_text())
print(
    {
        "base_accuracy": base["accuracy"],
        "merged_accuracy": merged["accuracy"],
        "delta": merged["accuracy"] - base["accuracy"],
    }
)
print("Classes:", merged["labels"])
print("Confusion matrix:", merged["confusion_matrix"])
print("Per-class metrics:", merged["classification_report"])

## Record the run
Save configs, split manifest, logs, software versions, and actual results. Add a separate human-authored test set before deployment. Do not publish an unauthenticated inference endpoint or upload credentials. See `training/README.md` for the local API hook; no public tunnel is created here.

In [ ]:
!pip freeze > training/data/environment.txt
from datetime import datetime, timezone

Path("training/data/run.json").write_text(
    json.dumps(
        {
            "finished_at": datetime.now(timezone.utc).isoformat(),
            "model": "Qwen/Qwen3-1.7B-Base",
            "method": "LoRA",
            "validation_examples": len(validation),
        },
        indent=2,
    )
)